# MoE Model Evaluation

## Part 0: Setup

In [1]:
# Install Dependencies
!pip install datasets transformers torch

Defaulting to user installation because normal site-packages is not writeable


Define some macros

In [2]:
import torch

context_length = 512
batch_size = 4
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
!pip install zstandard

Defaulting to user installation because normal site-packages is not writeable


## Part 1: Load Test Data

In [4]:
import os
import time
import logging
from tqdm import tqdm
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

# === Setup ===
logging.basicConfig(level=logging.WARNING)
os.makedirs("./data", exist_ok=True)

# === Constants ===
context_length = 1024
batch_size = 4
num_test_examples = 1000
tokenizer = AutoTokenizer.from_pretrained("stevensu123/gpt2-moe-topk")

# === Tokenization Function ===
def tokenize_function(example):
    return tokenizer(example["text"], return_attention_mask=False)

# === Group Texts into Chunks of Context Length ===
def group_texts(examples):
    concatenated = sum(examples["input_ids"], [])
    total_length = (len(concatenated) // context_length) * context_length
    input_ids = [concatenated[i:i + context_length] for i in range(0, total_length, context_length)]
    return {"input_ids": input_ids, "labels": input_ids}

# === Collect and Preprocess Test Data ===
def get_tokenized_test_data(num_examples=1000, max_retries=5, delay=2):
    stream = load_dataset("cerebras/SlimPajama-627B", split="test", streaming=True)
    iterator = iter(stream)
    examples = []
    pbar = tqdm(total=num_examples, desc=f"Collecting test examples")

    while len(examples) < num_examples:
        try:
            example = next(iterator)
            examples.append(example)
            pbar.update(1)
        except StopIteration:
            logging.warning(f"Reached end of stream before collecting {num_examples} examples.")
            break
        except Exception as e:
            retries = 0
            logging.warning(f"Error while collecting test example {len(examples)}: {e}")
            while retries < max_retries:
                try:
                    time.sleep(delay)
                    example = next(iterator)
                    examples.append(example)
                    pbar.update(1)
                    break
                except Exception as e_retry:
                    retries += 1
                    logging.warning(f"Retry {retries}/{max_retries} failed: {e_retry}")
            else:
                logging.warning(f"Skipping test example after {max_retries} retries.")
    pbar.close()

    raw_dataset = Dataset.from_dict({'text': [ex['text'] for ex in examples]})
    tokenized = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    grouped = tokenized.map(group_texts, batched=True)
    return grouped

# === Final Test Dataloader ===
test_dataset = get_tokenized_test_data(num_examples=num_test_examples)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map: 100%|██████████| 1000/1000 [00:02<00:00, 372.91 examples/s]


## Part 2: Loading Baseline

Load GPT2

In [5]:
# prompt: load GPT2-large

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load pre-trained GPT-2 model and tokenizer
baseline_model = GPT2LMHeadModel.from_pretrained('stevensu123/baseline')
baseline_tokenizer = GPT2Tokenizer.from_pretrained('stevensu123/baseline')

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_model.to(device)
baseline_model.eval()


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-35): 36 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=5120, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=5120)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1280, out_features=50257, bias=False)
)

Do a test inference run

In [6]:
# Example usage:
input_text = "Once upon a time"
input_ids = baseline_tokenizer.encode(input_text, return_tensors="pt").to(device)

# Generate text
output_ids = baseline_model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    early_stopping=True,
)

# Decode and print the output
output_text = baseline_tokenizer.decode(output_ids[0], skip_special_tokens=True)
output_text


/home/ubuntu/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'Once upon a time, there was a man named John. He was an old man, and he had a wife and a son. One day, he was walking down the street, when he saw a woman walking by. She was beautiful, but'

## Part 3: Loading GPT2-MoE Top-k

Load the Model

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Conv1D(nn.Module):
    def __init__(self, nf, nx):
        super().__init__()
        self.nf = nf
        w = torch.empty(nx, nf)
        nn.init.normal_(w, std=0.02)
        self.weight = nn.Parameter(w)
        self.bias = nn.Parameter(torch.zeros(nf))

    def forward(self, x):
        size_out = x.size()[:-1] + (self.nf,)
        x_view = x.view(-1, x.size(-1))
        result = x_view @ self.weight + self.bias
        return result.view(*size_out)

In [26]:
class ExpertLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.c_fc = Conv1D(hidden_dim, input_dim)
        self.c_proj = Conv1D(input_dim, hidden_dim)
        self.act = nn.GELU(approximate="tanh")

    def forward(self, x):
        hidden = self.c_fc(x)
        hidden = self.act(hidden)
        output = self.c_proj(hidden)
        return output

In [27]:
class MixtureOfExperts(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts=4, top_k=2, balance_loss_weight=0.01, layer_id=None):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.top_k = top_k
        self.balance_loss_weight = balance_loss_weight
        self.layer_id = layer_id

        self.experts = nn.ModuleList([
            ExpertLayer(input_dim, hidden_dim)
            for _ in range(num_experts)
        ])
        self.router = nn.Linear(input_dim, num_experts)
        self.register_buffer("usage_counter", torch.zeros(num_experts))

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        device = x.device

        router_logits = self.router(x)  # [B, T, E]
        routing_weights = F.softmax(router_logits, dim=-1)  # [B, T, E]

        topk_weights, topk_indices = torch.topk(routing_weights, self.top_k, dim=-1)  # [B, T, k]
        topk_weights /= topk_weights.sum(dim=-1, keepdim=True)  # Normalize

        x_flat = x.reshape(-1, d_model)  # [B*T, D]
        output_flat = torch.zeros_like(x_flat)
        expert_counts = torch.zeros(self.num_experts, device=device)

        topk_indices_flat = topk_indices.reshape(-1, self.top_k)
        topk_weights_flat = topk_weights.reshape(-1, self.top_k)

        for expert_id in range(self.num_experts):
            mask = (topk_indices_flat == expert_id)
            if not mask.any():
                continue

            token_indices, slot_indices = mask.nonzero(as_tuple=True)
            weights = topk_weights_flat[token_indices, slot_indices]
            x_selected = x_flat[token_indices]

            y_selected = self.experts[expert_id](x_selected)
            output_flat[token_indices] += weights.unsqueeze(1) * y_selected
            expert_counts[expert_id] = token_indices.numel()
            self.usage_counter[expert_id] += token_indices.numel()

        output = output_flat.view(batch_size, seq_len, d_model)

        num_tokens = batch_size * seq_len
        expert_fractions = expert_counts / (num_tokens * self.top_k)
        balance_loss = self.num_experts * torch.sum(expert_fractions ** 2)
        router_z_loss = torch.mean(torch.sum(router_logits ** 2, dim=-1))
        aux_loss = self.balance_loss_weight * (balance_loss + 0.001 * router_z_loss)
        self.aux_loss = aux_loss

        return output, aux_loss

In [ ]:
# class MixtureOfExpertsOptimized(nn.Module):
#     def __init__(self, input_dim, hidden_dim, num_experts=4, top_k=2, balance_loss_weight=0.01):
#         super().__init__()
#         self.input_dim = input_dim
#         self.hidden_dim = hidden_dim
#         self.num_experts = num_experts
#         self.top_k = top_k
#         self.balance_loss_weight = balance_loss_weight

#         self.experts = nn.ModuleList([
#             ExpertLayer(input_dim, hidden_dim)
#             for _ in range(num_experts)
#         ])
#         self.router = nn.Linear(input_dim, num_experts)
#         self.register_buffer("usage_counter", torch.zeros(num_experts))

#     def forward(self, x):
#         batch_size, seq_len, d_model = x.shape
#         device = x.device

#         router_logits = self.router(x)  # [B, T, E]
#         routing_weights = F.softmax(router_logits, dim=-1)  # [B, T, E]

#         topk_weights, topk_indices = torch.topk(routing_weights, self.top_k, dim=-1)  # [B, T, k]
#         topk_weights /= topk_weights.sum(dim=-1, keepdim=True)  # Normalize

#         B, T = batch_size, seq_len
#         x_flat = x.reshape(-1, d_model)  # [B*T, D]
#         topk_indices_flat = topk_indices.reshape(-1, self.top_k)  # [B*T, k]
#         topk_weights_flat = topk_weights.reshape(-1, self.top_k)  # [B*T, k]

#         expert_inputs = [[] for _ in range(self.num_experts)]
#         token_positions = [[] for _ in range(self.num_experts)]
#         weights_per_expert = [[] for _ in range(self.num_experts)]

#         for i in range(topk_indices_flat.shape[0]):
#             for j in range(self.top_k):
#                 expert_id = topk_indices_flat[i, j].item()
#                 expert_inputs[expert_id].append(x_flat[i])
#                 token_positions[expert_id].append(i)
#                 weights_per_expert[expert_id].append(topk_weights_flat[i, j].item())
#                 self.usage_counter[expert_id] += 1

#         output_flat = torch.zeros_like(x_flat)
#         expert_counts = torch.zeros(self.num_experts, device=device)

#         for expert_id in range(self.num_experts):
#             if expert_inputs[expert_id]:
#                 x_e = torch.stack(expert_inputs[expert_id])
#                 w_e = torch.tensor(weights_per_expert[expert_id], device=device).unsqueeze(-1)
#                 out_e = self.experts[expert_id](x_e)
#                 output_flat.index_add_(0,
#                                        torch.tensor(token_positions[expert_id], device=device),
#                                        out_e * w_e)
#                 expert_counts[expert_id] = len(token_positions[expert_id])

#         output = output_flat.view(batch_size, seq_len, d_model)

#         num_tokens = batch_size * seq_len
#         expert_fractions = expert_counts / (num_tokens * self.top_k)
#         balance_loss = self.num_experts * torch.sum(expert_fractions ** 2)
#         router_z_loss = torch.mean(torch.sum(router_logits ** 2, dim=-1))
#         aux_loss = self.balance_loss_weight * (balance_loss + 0.001 * router_z_loss)
#         self.aux_loss = aux_loss

#         return output, self.aux_loss


In [28]:
class GPT2MoEBlock(nn.Module):
    def __init__(self, original_block, num_experts=4, top_k=2, balance_loss_weight=0.01, layer_id=None):
        super().__init__()

        # Retain the same attention block and layer norm before the attention block and the MLP layer
        self.attn = original_block.attn
        self.ln_1 = original_block.ln_1
        self.ln_2 = original_block.ln_2
        hidden_size = original_block.ln_1.weight.size(0)
        intermediate_size = original_block.mlp.c_fc.weight.shape[1]

        # we will use the MoE block that we just defined to replace the MLPs in the attention block
        self.mlp = MixtureOfExperts(
            input_dim=hidden_size,
            hidden_dim=intermediate_size,
            num_experts=num_experts,
            top_k=top_k,
            balance_loss_weight=balance_loss_weight,
            layer_id=layer_id
        )

        # here, we loop through each expert block in the Mixture of expert layer and initialize their weights & biases
        for expert in self.mlp.experts:
            with torch.no_grad():

                # copy the weights of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.weight.data = original_block.mlp.c_fc.weight.clone().to(expert.c_fc.weight.device) / num_experts

                # copy the bias of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.bias.data = original_block.mlp.c_fc.bias.clone().to(expert.c_fc.bias.device) / num_experts

                # copy the weights of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.weight.data = original_block.mlp.c_proj.weight.clone().to(expert.c_proj.weight.device) / num_experts

                # copy the bias of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.bias.data = original_block.mlp.c_proj.bias.clone().to(expert.c_proj.bias.device) / num_experts

        with torch.no_grad():
            # initialize the router network's weights via random noise through a normal distribution
            nn.init.normal_(self.mlp.router.weight, mean=0.0, std=0.01)

        device = next(original_block.parameters()).device
        self = self.to(device)

    def forward(
        self,
        hidden_states,
        layer_past=None,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        use_cache=False,
        output_attentions=False,
        **kwargs
    ):

        # Residual connection. this is the input
        residual = hidden_states

        # Layer norm before passing input into attention
        hidden_states = self.ln_1(hidden_states)

        # arguments for the attention block
        attn_args = {
            "hidden_states": hidden_states,
            "layer_past": layer_past,
            "attention_mask": attention_mask,
            "head_mask": head_mask,
            "use_cache": use_cache,
            "output_attentions": output_attentions,
        }

        # passing in redundant arguments
        for key in kwargs:
            if key in ["position_ids", "position_embeddings"]:
                attn_args[key] = kwargs[key]

        # outputs from the attention block
        attn_outputs = self.attn(**attn_args)
        attn_output = attn_outputs[0]
        outputs = attn_outputs[1:]

        # Residual connection mechanism, adding the residual back to the input
        hidden_states = attn_output + residual

        # Second residual connection
        residual = hidden_states

        # Layer norm post attention block
        hidden_states = self.ln_2(hidden_states)

        # Passing into the mixture of experts layer
        hidden_states_moe, aux_loss = self.mlp(hidden_states)
        hidden_states = hidden_states_moe + residual

        # second residual connection
        if use_cache:
            outputs = (hidden_states,) + outputs
        else:
            outputs = (hidden_states,) + outputs[1:]
        return outputs

In [29]:
from transformers import GPT2LMHeadModel

class GPT2MoEModel(GPT2LMHeadModel):
    def __init__(self, config, num_experts=4, top_k=2):
        super().__init__(config)
        self.num_experts = num_experts
        self.top_k = top_k

    def inject_moe(self):
        total_layers = len(self.transformer.h)
        for idx in range(0, total_layers, 4):
            self.transformer.h[idx] = GPT2MoEBlock(
                self.transformer.h[idx], self.num_experts, self.top_k, layer_id=idx
            )

top_k_model = GPT2MoEModel.from_pretrained("stevensu123/gpt2-moe-topk")
top_k_tokenizer = AutoTokenizer.from_pretrained("stevensu123/gpt2-moe-topk")
top_k_model.inject_moe()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
top_k_model.to(device)
top_k_model.eval()

Some weights of the model checkpoint at stevensu123/gpt2-moe-topk were not used when initializing GPT2MoEModel: ['transformer.h.0.mlp.experts.0.c_fc.bias', 'transformer.h.0.mlp.experts.0.c_fc.weight', 'transformer.h.0.mlp.experts.0.c_proj.bias', 'transformer.h.0.mlp.experts.0.c_proj.weight', 'transformer.h.0.mlp.experts.1.c_fc.bias', 'transformer.h.0.mlp.experts.1.c_fc.weight', 'transformer.h.0.mlp.experts.1.c_proj.bias', 'transformer.h.0.mlp.experts.1.c_proj.weight', 'transformer.h.0.mlp.experts.2.c_fc.bias', 'transformer.h.0.mlp.experts.2.c_fc.weight', 'transformer.h.0.mlp.experts.2.c_proj.bias', 'transformer.h.0.mlp.experts.2.c_proj.weight', 'transformer.h.0.mlp.experts.3.c_fc.bias', 'transformer.h.0.mlp.experts.3.c_fc.weight', 'transformer.h.0.mlp.experts.3.c_proj.bias', 'transformer.h.0.mlp.experts.3.c_proj.weight', 'transformer.h.0.mlp.router.bias', 'transformer.h.0.mlp.router.weight', 'transformer.h.12.mlp.experts.0.c_fc.bias', 'transformer.h.12.mlp.experts.0.c_fc.weight', 'tran

GPT2MoEModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0): GPT2MoEBlock(
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): MixtureOfExperts(
          (experts): ModuleList(
            (0-3): 4 x ExpertLayer(
              (c_fc): Conv1D()
              (c_proj): Conv1D()
              (act): GELU(approximate='tanh')
            )
          )
          (router): Linear(in_features=1280, out_features=4, bias=True)
        )
      )
      (1-3): 3 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_aff

Do a test inference run

In [23]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Set pad_token_id to eos_token_id if not already set
top_k_model.config.pad_token_id = top_k_model.config.eos_token_id

# Prepare input
input_text = "Once upon a time"
input_ids = top_k_tokenizer.encode(input_text, return_tensors='pt').to(device)

# Generate text
output_ids = top_k_model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    early_stopping=True
)

# Decode and print the output
output_text = top_k_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)


/home/ubuntu/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time, the world of the internet is a very of its day. You can surf the web, read news, watch videos, and even if you're in the mood, you can even play some game of Minecraft.
But what


## Part 4: Loading GPT2-MoE Top-p

Load the model

In [33]:
class Conv1D(nn.Module):
    def __init__(self, nf, nx):
        super().__init__()
        self.nf = nf
        w = torch.empty(nx, nf)
        nn.init.normal_(w, std=0.02)
        self.weight = nn.Parameter(w)
        self.bias = nn.Parameter(torch.zeros(nf))

    def forward(self, x):
        size_out = x.size()[:-1] + (self.nf,)
        x_view = x.view(-1, x.size(-1))
        result = x_view @ self.weight + self.bias
        return result.view(*size_out)

In [34]:
class ExpertLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.c_fc = Conv1D(hidden_dim, input_dim)
        self.c_proj = Conv1D(input_dim, hidden_dim)
        self.act = nn.GELU(approximate="tanh")

    def forward(self, x):
        hidden = self.c_fc(x)
        hidden = self.act(hidden)
        output = self.c_proj(hidden)
        return output

In [35]:
class MixtureOfExperts(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts=4, top_p=0.8, balance_loss_weight=0.01, layer_id=None):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.top_p = top_p
        self.balance_loss_weight = balance_loss_weight
        self.layer_id = layer_id

        self.experts = nn.ModuleList([
            ExpertLayer(input_dim, hidden_dim)
            for _ in range(num_experts)
        ])
        self.router = nn.Linear(input_dim, num_experts)
        self.register_buffer("usage_counter", torch.zeros(num_experts))

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        device = x.device

        router_logits = self.router(x)  # [B, T, E]
        routing_probs = F.softmax(router_logits, dim=-1)  # [B, T, E]

        sorted_probs, sorted_indices = torch.sort(routing_probs, dim=-1, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        top_p_mask = cumulative_probs <= self.top_p
        top_p_mask[..., 0] = 1  # Ensure at least one expert is selected

        expert_mask = torch.zeros_like(routing_probs, dtype=torch.bool)
        expert_mask.scatter_(-1, sorted_indices, top_p_mask)

        selected_routing_probs = routing_probs * expert_mask
        normalized_probs = selected_routing_probs / (selected_routing_probs.sum(dim=-1, keepdim=True) + 1e-9)

        x_flat = x.reshape(-1, d_model)
        output_flat = torch.zeros_like(x_flat)
        expert_counts = torch.zeros(self.num_experts, device=device)

        for expert_id in range(self.num_experts):
            mask = expert_mask[..., expert_id].reshape(-1)
            if not mask.any():
                continue

            token_indices = mask.nonzero(as_tuple=False).squeeze(1)
            x_selected = x_flat[token_indices]
            weights = normalized_probs[..., expert_id].reshape(-1)[token_indices]

            y_selected = self.experts[expert_id](x_selected)
            output_flat[token_indices] += weights.unsqueeze(1) * y_selected
            expert_counts[expert_id] = token_indices.numel()
            self.usage_counter[expert_id] += token_indices.numel()

        output = output_flat.view(batch_size, seq_len, d_model)

        num_tokens = batch_size * seq_len
        expert_fractions = expert_counts / num_tokens
        balance_loss = self.num_experts * torch.sum(expert_fractions ** 2)
        router_z_loss = torch.mean(torch.sum(router_logits ** 2, dim=-1))
        aux_loss = self.balance_loss_weight * (balance_loss + 0.001 * router_z_loss)
        self.aux_loss = aux_loss

        return output, aux_loss

In [36]:
class GPT2MoEBlock(nn.Module):
    def __init__(self, original_block, num_experts=4, top_p=0.8, balance_loss_weight=0.01, layer_id=None):
        super().__init__()

        # Retain the same attention block and layer norm before the attention block and the MLP layer
        self.attn = original_block.attn
        self.ln_1 = original_block.ln_1
        self.ln_2 = original_block.ln_2
        hidden_size = original_block.ln_1.weight.size(0)
        intermediate_size = original_block.mlp.c_fc.weight.shape[1]

        # we will use the MoE block that we just defined to replace the MLPs in the attention block
        self.mlp = MixtureOfExperts(
            input_dim=hidden_size,
            hidden_dim=intermediate_size,
            num_experts=num_experts,
            top_p=top_p,
            balance_loss_weight=balance_loss_weight,
            layer_id=layer_id
        )

        # here, we loop through each expert block in the Mixture of expert layer and initialize their weights & biases
        for expert in self.mlp.experts:
            with torch.no_grad():

                # copy the weights of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.weight.data = original_block.mlp.c_fc.weight.clone().to(expert.c_fc.weight.device) / num_experts

                # copy the bias of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.bias.data = original_block.mlp.c_fc.bias.clone().to(expert.c_fc.bias.device) / num_experts

                # copy the weights of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.weight.data = original_block.mlp.c_proj.weight.clone().to(expert.c_proj.weight.device) / num_experts

                # copy the bias of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.bias.data = original_block.mlp.c_proj.bias.clone().to(expert.c_proj.bias.device) / num_experts

        with torch.no_grad():
            # initialize the router network's weights via random noise through a normal distribution
            nn.init.normal_(self.mlp.router.weight, mean=0.0, std=0.01)

        device = next(original_block.parameters()).device
        self = self.to(device)

    def forward(
        self,
        hidden_states,
        layer_past=None,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        use_cache=False,
        output_attentions=False,
        **kwargs
    ):

        # Residual connection. this is the input
        residual = hidden_states

        # Layer norm before passing input into attention
        hidden_states = self.ln_1(hidden_states)

        # arguments for the attention block
        attn_args = {
            "hidden_states": hidden_states,
            "layer_past": layer_past,
            "attention_mask": attention_mask,
            "head_mask": head_mask,
            "use_cache": use_cache,
            "output_attentions": output_attentions,
        }

        # passing in redundant arguments
        for key in kwargs:
            if key in ["position_ids", "position_embeddings"]:
                attn_args[key] = kwargs[key]

        # outputs from the attention block
        attn_outputs = self.attn(**attn_args)
        attn_output = attn_outputs[0]
        outputs = attn_outputs[1:]

        # Residual connection mechanism, adding the residual back to the input
        hidden_states = attn_output + residual

        # Second residual connection
        residual = hidden_states

        # Layer norm post attention block
        hidden_states = self.ln_2(hidden_states)

        # Passing into the mixture of experts layer
        hidden_states_moe, aux_loss = self.mlp(hidden_states)
        hidden_states = hidden_states_moe + residual

        # second residual connection
        if use_cache:
            outputs = (hidden_states,) + outputs
        else:
            outputs = (hidden_states,) + outputs[1:]
        return outputs

In [ ]:
from transformers import GPT2LMHeadModel

class GPT2MoEModel(GPT2LMHeadModel):
    def __init__(self, config, num_experts=4, top_p=0.8):
        super().__init__(config)
        self.num_experts = num_experts
        self.top_p = top_p

    def inject_moe(self):
        total_layers = len(self.transformer.h)
        for idx in range(0, total_layers, 4):
            self.transformer.h[idx] = GPT2MoEBlock(
                self.transformer.h[idx], self.num_experts, self.top_p, layer_id=idx
            )

top_p_model = GPT2MoEModel.from_pretrained("stevensu123/gpt2-moe-topp")
top_p_tokenizer = AutoTokenizer.from_pretrained("stevensu123/gpt2-moe-topp")
top_p_model.inject_moe()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
top_p_model.to(device)
top_p_model.eval()

Some weights of the model checkpoint at stevensu123/gpt2-moe-topp were not used when initializing GPT2MoEModel: ['transformer.h.0.mlp.experts.0.c_fc.bias', 'transformer.h.0.mlp.experts.0.c_fc.weight', 'transformer.h.0.mlp.experts.0.c_proj.bias', 'transformer.h.0.mlp.experts.0.c_proj.weight', 'transformer.h.0.mlp.experts.1.c_fc.bias', 'transformer.h.0.mlp.experts.1.c_fc.weight', 'transformer.h.0.mlp.experts.1.c_proj.bias', 'transformer.h.0.mlp.experts.1.c_proj.weight', 'transformer.h.0.mlp.experts.2.c_fc.bias', 'transformer.h.0.mlp.experts.2.c_fc.weight', 'transformer.h.0.mlp.experts.2.c_proj.bias', 'transformer.h.0.mlp.experts.2.c_proj.weight', 'transformer.h.0.mlp.experts.3.c_fc.bias', 'transformer.h.0.mlp.experts.3.c_fc.weight', 'transformer.h.0.mlp.experts.3.c_proj.bias', 'transformer.h.0.mlp.experts.3.c_proj.weight', 'transformer.h.0.mlp.router.bias', 'transformer.h.0.mlp.router.weight', 'transformer.h.12.mlp.experts.0.c_fc.bias', 'transformer.h.12.mlp.experts.0.c_fc.weight', 'tran

GPT2MoEModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0): GPT2MoEBlock(
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): MixtureOfExperts(
          (experts): ModuleList(
            (0-3): 4 x ExpertLayer(
              (c_fc): Conv1D()
              (c_proj): Conv1D()
              (act): GELU(approximate='tanh')
            )
          )
          (router): Linear(in_features=1280, out_features=4, bias=True)
        )
      )
      (1-3): 3 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_aff

Do a test inference

In [38]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer


# Set pad_token_id to eos_token_id if not already set
top_p_model.config.pad_token_id = top_p_model.config.eos_token_id

# Prepare input
input_text = "Once upon a time"
input_ids = top_p_tokenizer.encode(input_text, return_tensors='pt').to(device)

# Generate text
output_ids = top_p_model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=1,
    no_repeat_ngram_size=2,
    early_stopping=True
)

# Decode and print the output
output_text = top_p_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output_text)


/home/ubuntu/.local/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Once upon a time, I was in a bar, and I had a few time time and the bartender was a one and he was the one was I and then was and and was.
I was sitting in the bar and she was there was


## Part 5: Loading GPT2-MoE Top-any

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Conv1D(nn.Module):
    def __init__(self, nf, nx):
        super().__init__()
        self.nf = nf
        w = torch.empty(nx, nf)
        nn.init.normal_(w, std=0.02)
        self.weight = nn.Parameter(w)
        self.bias = nn.Parameter(torch.zeros(nf))

    def forward(self, x):
        size_out = x.size()[:-1] + (self.nf,)
        x_view = x.view(-1, x.size(-1))
        result = x_view @ self.weight + self.bias
        return result.view(*size_out)

In [20]:
class ExpertLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.c_fc = Conv1D(hidden_dim, input_dim)
        self.c_proj = Conv1D(input_dim, hidden_dim)
        self.act = nn.GELU(approximate="tanh")

    def forward(self, x):
        hidden = self.c_fc(x)
        hidden = self.act(hidden)
        output = self.c_proj(hidden)
        return output

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MixtureOfExperts(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts=12, balance_loss_weight=0.01):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.balance_loss_weight = balance_loss_weight

        self.experts = nn.ModuleList([
            ExpertLayer(input_dim, hidden_dim)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(input_dim, num_experts)
        self.thresholds = nn.Parameter(torch.zeros(num_experts))  # Trainable thresholds
        self.register_buffer("usage_counter", torch.zeros(num_experts))

    def forward(self, x, is_training=True):
        batch_size, seq_len, d_model = x.shape
        device = x.device
        x_flat = x.view(-1, d_model)  # [B*T, D]
        router_logits = self.router(x)  # [B, T, E]
        routing_scores = torch.sigmoid(router_logits)  # [B, T, E]

        thresholds = torch.sigmoid(self.thresholds)  # [E]
        thresholds = thresholds.unsqueeze(0).unsqueeze(0)  # [1, 1, E]

        mask = (routing_scores >= thresholds).float()  # [B, T, E]

        # Ensure at least one expert is used per token (fallback to top-1 if none selected)
        none_selected = (mask.sum(dim=-1) == 0)  # [B, T]
        if not is_training:
            top1_idx = torch.argmax(routing_scores, dim=-1, keepdim=True)  # [B, T, 1]
            fallback_mask = torch.zeros_like(mask)
            fallback_mask.scatter_(-1, top1_idx, 1.0)
            mask = torch.where(none_selected.unsqueeze(-1), fallback_mask, mask)

        mask_flat = mask.view(-1, self.num_experts)  # [B*T, E]
        scores_flat = routing_scores.view(-1, self.num_experts) * mask_flat  # [B*T, E]
        norm_scores_flat = scores_flat / (scores_flat.sum(dim=-1, keepdim=True) + 1e-9)  # normalize

        output_flat = torch.zeros_like(x_flat)  # [B*T, D]
        expert_counts = torch.zeros(self.num_experts, device=device)

        for e in range(self.num_experts):
            selected = (mask_flat[:, e] > 0)  # [B*T]
            if selected.sum() == 0:
                continue

            x_e = x_flat[selected]  # [N_e, D]
            y_e = self.experts[e](x_e)  # [N_e, D]
            w_e = norm_scores_flat[selected, e].unsqueeze(-1)  # [N_e, 1]
            output_flat[selected] += w_e * y_e
            expert_counts[e] = selected.sum()
            self.usage_counter[e] += selected.sum().item()

        output = output_flat.view(batch_size, seq_len, d_model)

        # === Load balancing loss ===
        num_tokens = batch_size * seq_len
        expert_fractions = expert_counts / num_tokens
        balance_loss = self.num_experts * torch.sum(expert_fractions ** 2)

        # === Router z-loss ===
        router_z_loss = torch.mean(torch.sum(router_logits ** 2, dim=-1))

        aux_loss = self.balance_loss_weight * (balance_loss + 0.001 * router_z_loss)
        self.aux_loss = aux_loss

        return output, self.aux_loss


In [22]:
class GPT2MoEBlock(nn.Module):
    def __init__(self, original_block, num_experts=4, balance_loss_weight=0.01):
        super().__init__()

        # Retain the same attention block and layer norm before the attention block and the MLP layer
        self.attn = original_block.attn
        self.ln_1 = original_block.ln_1
        self.ln_2 = original_block.ln_2
        hidden_size = original_block.ln_1.weight.size(0)
        intermediate_size = original_block.mlp.c_fc.weight.shape[1]

        # we will use the MoE block that we just defined to replace the MLPs in the attention block
        self.mlp = MixtureOfExperts(
            input_dim=hidden_size,
            hidden_dim=intermediate_size,
            num_experts=num_experts,
            balance_loss_weight=balance_loss_weight
        )

        # here, we loop through each expert block in the Mixture of expert layer and initialize their weights & biases
        for expert in self.mlp.experts:
            with torch.no_grad():

                # copy the weights of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.weight.data = original_block.mlp.c_fc.weight.clone().to(expert.c_fc.weight.device) / num_experts

                # copy the bias of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.bias.data = original_block.mlp.c_fc.bias.clone().to(expert.c_fc.bias.device) / num_experts

                # copy the weights of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.weight.data = original_block.mlp.c_proj.weight.clone().to(expert.c_proj.weight.device) / num_experts

                # copy the bias of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.bias.data = original_block.mlp.c_proj.bias.clone().to(expert.c_proj.bias.device) / num_experts

        with torch.no_grad():
            # initialize the router network's weights via random noise through a normal distribution
            nn.init.normal_(self.mlp.router.weight, mean=0.0, std=0.01)

        device = next(original_block.parameters()).device
        self = self.to(device)

    def forward(
        self,
        hidden_states,
        layer_past=None,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        use_cache=False,
        output_attentions=False,
        **kwargs
    ):

        # Residual connection. this is the input
        residual = hidden_states

        # Layer norm before passing input into attention
        hidden_states = self.ln_1(hidden_states)

        # arguments for the attention block
        attn_args = {
            "hidden_states": hidden_states,
            "layer_past": layer_past,
            "attention_mask": attention_mask,
            "head_mask": head_mask,
            "use_cache": use_cache,
            "output_attentions": output_attentions,
        }

        # passing in redundant arguments
        for key in kwargs:
            if key in ["position_ids", "position_embeddings"]:
                attn_args[key] = kwargs[key]

        # outputs from the attention block
        attn_outputs = self.attn(**attn_args)
        attn_output = attn_outputs[0]
        outputs = attn_outputs[1:]

        # Residual connection mechanism, adding the residual back to the input
        hidden_states = attn_output + residual

        # Second residual connection
        residual = hidden_states

        # Layer norm post attention block
        hidden_states = self.ln_2(hidden_states)

        # Passing into the mixture of experts layer
        hidden_states_moe, aux_loss = self.mlp(hidden_states)
        hidden_states = hidden_states_moe + residual

        # second residual connection
        if use_cache:
            outputs = (hidden_states,) + outputs
        else:
            outputs = (hidden_states,) + outputs[1:]
        return outputs

In [23]:
from transformers import GPT2LMHeadModel

class GPT2MoEModel(GPT2LMHeadModel):
    def __init__(self, config, num_experts=4):
        super().__init__(config)
        self.num_experts = num_experts

    def inject_moe(self):
        total_layers = len(self.transformer.h)
        for idx in range(0, total_layers, 4):
            self.transformer.h[idx] = GPT2MoEBlock(
                self.transformer.h[idx], self.num_experts
            )

top_any_model = GPT2MoEModel.from_pretrained("tarunyaa/gpt2-moe-topany")
top_any_tokenizer = AutoTokenizer.from_pretrained("tarunyaa/gpt2-moe-topany")
top_any_model.inject_moe()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
top_any_model.to(device)
top_any_model.eval()

Some weights of the model checkpoint at tarunyaa/gpt2-moe-topany were not used when initializing GPT2MoEModel: ['transformer.h.0.mlp.experts.0.c_fc.bias', 'transformer.h.0.mlp.experts.0.c_fc.weight', 'transformer.h.0.mlp.experts.0.c_proj.bias', 'transformer.h.0.mlp.experts.0.c_proj.weight', 'transformer.h.0.mlp.experts.1.c_fc.bias', 'transformer.h.0.mlp.experts.1.c_fc.weight', 'transformer.h.0.mlp.experts.1.c_proj.bias', 'transformer.h.0.mlp.experts.1.c_proj.weight', 'transformer.h.0.mlp.experts.2.c_fc.bias', 'transformer.h.0.mlp.experts.2.c_fc.weight', 'transformer.h.0.mlp.experts.2.c_proj.bias', 'transformer.h.0.mlp.experts.2.c_proj.weight', 'transformer.h.0.mlp.experts.3.c_fc.bias', 'transformer.h.0.mlp.experts.3.c_fc.weight', 'transformer.h.0.mlp.experts.3.c_proj.bias', 'transformer.h.0.mlp.experts.3.c_proj.weight', 'transformer.h.0.mlp.router.bias', 'transformer.h.0.mlp.router.weight', 'transformer.h.0.mlp.thresholds', 'transformer.h.12.mlp.experts.0.c_fc.bias', 'transformer.h.12.

GPT2MoEModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0): GPT2MoEBlock(
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): MixtureOfExperts(
          (experts): ModuleList(
            (0-3): 4 x ExpertLayer(
              (c_fc): Conv1D()
              (c_proj): Conv1D()
              (act): GELU(approximate='tanh')
            )
          )
          (router): Linear(in_features=1280, out_features=4, bias=True)
        )
      )
      (1-3): 3 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_aff

## Part 6: Loading GPT2-MoE Entropy Adaptive

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Conv1D(nn.Module):
    def __init__(self, nf, nx):
        super().__init__()
        self.nf = nf
        w = torch.empty(nx, nf)
        nn.init.normal_(w, std=0.02)
        self.weight = nn.Parameter(w)
        self.bias = nn.Parameter(torch.zeros(nf))

    def forward(self, x):
        size_out = x.size()[:-1] + (self.nf,)
        x_view = x.view(-1, x.size(-1))
        result = x_view @ self.weight + self.bias
        return result.view(*size_out)

In [25]:
class ExpertLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.c_fc = Conv1D(hidden_dim, input_dim)
        self.c_proj = Conv1D(input_dim, hidden_dim)
        self.act = nn.GELU(approximate="tanh")

    def forward(self, x):
        hidden = self.c_fc(x)
        hidden = self.act(hidden)
        output = self.c_proj(hidden)
        return output

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MixtureOfExperts(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts=12, min_k=1, max_k=4, balance_loss_weight=0.01):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_experts = num_experts
        self.min_k = min_k
        self.max_k = max_k
        self.balance_loss_weight = balance_loss_weight

        self.experts = nn.ModuleList([
            ExpertLayer(input_dim, hidden_dim)
            for _ in range(num_experts)
        ])

        self.router = nn.Linear(input_dim, num_experts)
        self.register_buffer("usage_counter", torch.zeros(num_experts))

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        device = x.device

        # === Step 1: Compute routing scores ===
        router_logits = self.router(x)  # [B, T, E]
        routing_weights = F.softmax(router_logits, dim=-1)  # [B, T, E]

        # === Step 2: Compute entropy and determine k per token ===
        entropy = -(routing_weights * torch.log(routing_weights + 1e-9)).sum(dim=-1)  # [B, T]
        max_entropy = torch.log(torch.tensor(float(self.num_experts), device=device))
        norm_entropy = entropy / max_entropy
        k_per_token = (norm_entropy * (self.max_k - self.min_k)).ceil().long() + self.min_k  # [B, T]

        # === Step 3: Sort expert scores and prepare selection ===
        sorted_weights, sorted_indices = torch.sort(routing_weights, dim=-1, descending=True)  # [B, T, E]
        selected_weights = sorted_weights[:, :, :self.max_k]  # [B, T, max_k]
        selected_indices = sorted_indices[:, :, :self.max_k]  # [B, T, max_k]

        token_k = k_per_token.clamp(max=self.max_k).unsqueeze(-1)  # [B, T, 1]
        range_k = torch.arange(self.max_k, device=device).view(1, 1, -1)
        mask = (range_k < token_k)  # [B, T, max_k]

        selected_weights = selected_weights * mask
        selected_weights = selected_weights / (selected_weights.sum(dim=-1, keepdim=True) + 1e-8)  # [B, T, max_k]

        # === Step 4: Apply expert outputs ===
        x_flat = x.view(-1, d_model)  # [B*T, D]
        output_flat = torch.zeros_like(x_flat)
        expert_counts = torch.zeros(self.num_experts, device=device)

        selected_indices_flat = selected_indices.view(-1, self.max_k)  # [B*T, max_k]
        selected_weights_flat = selected_weights.view(-1, self.max_k)  # [B*T, max_k]
        mask_flat = mask.view(-1, self.max_k)

        for expert_id in range(self.num_experts):
            for k in range(self.max_k):
                mask_k = (selected_indices_flat[:, k] == expert_id) & mask_flat[:, k]  # [B*T]
                if mask_k.sum() == 0:
                    continue

                token_indices = torch.nonzero(mask_k).squeeze(-1)
                x_selected = x_flat[token_indices]
                weights = selected_weights_flat[token_indices, k]
                y_selected = self.experts[expert_id](x_selected)
                output_flat[token_indices] += weights.unsqueeze(-1) * y_selected
                expert_counts[expert_id] += len(token_indices)
                self.usage_counter[expert_id] += token_indices.numel()

        output = output_flat.view(batch_size, seq_len, d_model)

        # === Step 5: Load balancing loss ===
        num_tokens = batch_size * seq_len
        expert_fractions = expert_counts / (num_tokens * self.max_k)
        balance_loss = self.num_experts * torch.sum(expert_fractions ** 2)
        router_z_loss = torch.mean(torch.sum(router_logits ** 2, dim=-1))
        aux_loss = self.balance_loss_weight * (balance_loss + 0.001 * router_z_loss)

        return output, aux_loss


In [27]:
class GPT2MoEBlock(nn.Module):
    def __init__(self, original_block, num_experts=4, min_k=1, max_k=4, balance_loss_weight=0.01):
        super().__init__()

        # Retain the same attention block and layer norm before the attention block and the MLP layer
        self.attn = original_block.attn
        self.ln_1 = original_block.ln_1
        self.ln_2 = original_block.ln_2
        hidden_size = original_block.ln_1.weight.size(0)
        intermediate_size = original_block.mlp.c_fc.weight.shape[1]

        # we will use the MoE block that we just defined to replace the MLPs in the attention block
        self.mlp = MixtureOfExperts(
            input_dim=hidden_size,
            hidden_dim=intermediate_size,
            num_experts=num_experts,
            min_k=min_k, 
            max_k=max_k,
            balance_loss_weight=balance_loss_weight
        )

        # here, we loop through each expert block in the Mixture of expert layer and initialize their weights & biases
        for expert in self.mlp.experts:
            with torch.no_grad():

                # copy the weights of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.weight.data = original_block.mlp.c_fc.weight.clone().to(expert.c_fc.weight.device) / num_experts

                # copy the bias of the original MLP block's up projection layer to each expert and normalize by division of the number of experts
                expert.c_fc.bias.data = original_block.mlp.c_fc.bias.clone().to(expert.c_fc.bias.device) / num_experts

                # copy the weights of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.weight.data = original_block.mlp.c_proj.weight.clone().to(expert.c_proj.weight.device) / num_experts

                # copy the bias of the original MLP block's down projection layer to each expert and normalize by division of the number of experts
                expert.c_proj.bias.data = original_block.mlp.c_proj.bias.clone().to(expert.c_proj.bias.device) / num_experts

        with torch.no_grad():
            # initialize the router network's weights via random noise through a normal distribution
            nn.init.normal_(self.mlp.router.weight, mean=0.0, std=0.01)

        device = next(original_block.parameters()).device
        self = self.to(device)

    def forward(
        self,
        hidden_states,
        layer_past=None,
        attention_mask=None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        use_cache=False,
        output_attentions=False,
        **kwargs
    ):

        # Residual connection. this is the input
        residual = hidden_states

        # Layer norm before passing input into attention
        hidden_states = self.ln_1(hidden_states)

        # arguments for the attention block
        attn_args = {
            "hidden_states": hidden_states,
            "layer_past": layer_past,
            "attention_mask": attention_mask,
            "head_mask": head_mask,
            "use_cache": use_cache,
            "output_attentions": output_attentions,
        }

        # passing in redundant arguments
        for key in kwargs:
            if key in ["position_ids", "position_embeddings"]:
                attn_args[key] = kwargs[key]

        # outputs from the attention block
        attn_outputs = self.attn(**attn_args)
        attn_output = attn_outputs[0]
        outputs = attn_outputs[1:]

        # Residual connection mechanism, adding the residual back to the input
        hidden_states = attn_output + residual

        # Second residual connection
        residual = hidden_states

        # Layer norm post attention block
        hidden_states = self.ln_2(hidden_states)

        # Passing into the mixture of experts layer
        hidden_states_moe, aux_loss = self.mlp(hidden_states)
        hidden_states = hidden_states_moe + residual

        # second residual connection
        if use_cache:
            outputs = (hidden_states,) + outputs
        else:
            outputs = (hidden_states,) + outputs[1:]
        return outputs

In [28]:
from transformers import GPT2LMHeadModel

class GPT2MoEModel(GPT2LMHeadModel):
    def __init__(self, config, num_experts=4):
        super().__init__(config)
        self.num_experts = num_experts

    def inject_moe(self):
        total_layers = len(self.transformer.h)
        for idx in range(0, total_layers, 4):
            self.transformer.h[idx] = GPT2MoEBlock(
                self.transformer.h[idx], self.num_experts
            )

entropy_model = GPT2MoEModel.from_pretrained("tarunyaa/gpt2-moe-entropy")
entropy_tokenizer = AutoTokenizer.from_pretrained("tarunyaa/gpt2-moe-entropy")
entropy_model.inject_moe()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
entropy_model.to(device)
entropy_model.eval()

Some weights of the model checkpoint at tarunyaa/gpt2-moe-entropy were not used when initializing GPT2MoEModel: ['transformer.h.0.mlp.experts.0.c_fc.bias', 'transformer.h.0.mlp.experts.0.c_fc.weight', 'transformer.h.0.mlp.experts.0.c_proj.bias', 'transformer.h.0.mlp.experts.0.c_proj.weight', 'transformer.h.0.mlp.experts.1.c_fc.bias', 'transformer.h.0.mlp.experts.1.c_fc.weight', 'transformer.h.0.mlp.experts.1.c_proj.bias', 'transformer.h.0.mlp.experts.1.c_proj.weight', 'transformer.h.0.mlp.experts.2.c_fc.bias', 'transformer.h.0.mlp.experts.2.c_fc.weight', 'transformer.h.0.mlp.experts.2.c_proj.bias', 'transformer.h.0.mlp.experts.2.c_proj.weight', 'transformer.h.0.mlp.experts.3.c_fc.bias', 'transformer.h.0.mlp.experts.3.c_fc.weight', 'transformer.h.0.mlp.experts.3.c_proj.bias', 'transformer.h.0.mlp.experts.3.c_proj.weight', 'transformer.h.0.mlp.router.bias', 'transformer.h.0.mlp.router.weight', 'transformer.h.12.mlp.experts.0.c_fc.bias', 'transformer.h.12.mlp.experts.0.c_fc.weight', 'tran

GPT2MoEModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1280)
    (wpe): Embedding(1024, 1280)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0): GPT2MoEBlock(
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3840, nx=1280)
          (c_proj): Conv1D(nf=1280, nx=1280)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (mlp): MixtureOfExperts(
          (experts): ModuleList(
            (0-3): 4 x ExpertLayer(
              (c_fc): Conv1D()
              (c_proj): Conv1D()
              (act): GELU(approximate='tanh')
            )
          )
          (router): Linear(in_features=1280, out_features=4, bias=True)
        )
      )
      (1-3): 3 x GPT2Block(
        (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_aff

## Part 7: Evaluations

In [11]:
import torch
import time
import psutil
import os
from tqdm import tqdm

def evaluate_model(model, test_dataloader, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    inference_times = []
    peak_memories = []
    average_memories = []

    process = psutil.Process(os.getpid())

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating"):
            input_ids = torch.stack(batch["input_ids"]).to(device)
            labels = torch.stack(batch["labels"]).to(device)

            # Memory before inference
            if device == "cuda":
                torch.cuda.reset_peak_memory_stats()
            mem_before = process.memory_info().rss

            start_time = time.perf_counter()
            outputs = model(input_ids=input_ids, labels=labels)
            end_time = time.perf_counter()

            loss = outputs.loss
            batch_tokens = input_ids.numel()
            total_loss += loss.item() * batch_tokens
            total_tokens += batch_tokens

            mem_after = process.memory_info().rss
            peak_mem = (
                torch.cuda.max_memory_allocated(device=device)
                if device == "cuda"
                else mem_after
            )

            elapsed = end_time - start_time

            # Log stats
            inference_times.append(elapsed)
            average_memories.append((mem_after + mem_before) / 2)
            peak_memories.append(peak_mem)

    average_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(average_loss))

    print(f"\n🧪 Evaluation Results:")
    print(f"  - Average Loss: {average_loss:.4f}")
    print(f"  - Perplexity: {perplexity:.2f}")
    print(f"  - Total Inference Time: {sum(inference_times):.2f}s")
    print(f"  - Average Time per Batch: {sum(inference_times)/len(inference_times):.4f}s")
    print(f"  - Average Time per Token: {sum(inference_times)/total_tokens * 1000:.4f} ms")
    print(f"  - Average Peak Memory per Batch: {sum(peak_memories) / len(peak_memories) / 1024**2:.2f} MB")
    print(f"  - Average Memory Used per Batch: {sum(average_memories) / len(average_memories) / 1024**2:.2f} MB")

    return {
        "loss": average_loss,
        "perplexity": perplexity.item(),
        "inference_times": inference_times,
        "peak_memories": peak_memories,
        "average_memories": average_memories,
    }


In [12]:
import torch
import time
import psutil
import os
from tqdm import tqdm
from torch.profiler import profile, ProfilerActivity

def evaluate_model(model, test_dataloader, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    inference_times = []
    peak_memories = []
    average_memories = []
    total_flops = 0.0  # New

    process = psutil.Process(os.getpid())

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating"):
            input_ids = torch.stack(batch["input_ids"]).to(device)
            labels = torch.stack(batch["labels"]).to(device)

            # Memory before inference
            if device == "cuda":
                torch.cuda.reset_peak_memory_stats()
            mem_before = process.memory_info().rss

            start_time = time.perf_counter()
            
            # === FLOPs profiler ===
            with profile(
                activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA] if device == "cuda" else [ProfilerActivity.CPU],
                with_flops=True,
                record_shapes=True,
                profile_memory=False,
                with_stack=False,
                use_cuda=(device == "cuda"),
            ) as prof:
                outputs = model(input_ids=input_ids, labels=labels)
            
            end_time = time.perf_counter()

            # Extract FLOPs
            flops = sum([e.flops for e in prof.key_averages() if e.flops is not None])
            total_flops += flops

            loss = outputs.loss
            batch_tokens = input_ids.numel()
            total_loss += loss.item() * batch_tokens
            total_tokens += batch_tokens

            mem_after = process.memory_info().rss
            peak_mem = (
                torch.cuda.max_memory_allocated(device=device)
                if device == "cuda"
                else mem_after
            )

            elapsed = end_time - start_time

            # Log stats
            inference_times.append(elapsed)
            average_memories.append((mem_after + mem_before) / 2)
            peak_memories.append(peak_mem)

    average_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(average_loss))

    print(f"\n🧪 Evaluation Results:")
    print(f"  - Average Loss: {average_loss:.4f}")
    print(f"  - Perplexity: {perplexity:.2f}")
    print(f"  - Total Inference Time: {sum(inference_times):.2f}s")
    print(f"  - Average Time per Batch: {sum(inference_times)/len(inference_times):.4f}s")
    print(f"  - Average Time per Token: {sum(inference_times)/total_tokens * 1000:.4f} ms")
    print(f"  - Average Peak Memory per Batch: {sum(peak_memories) / len(peak_memories) / 1024**2:.2f} MB")
    print(f"  - Average Memory Used per Batch: {sum(average_memories) / len(average_memories) / 1024**2:.2f} MB")
    print(f"  - Total FLOPs: {total_flops / 1e9:.2f} GFLOPs")
    print(f"  - Average FLOPs per Token: {total_flops / total_tokens / 1e6:.2f} MFLOPs")

    return {
        "loss": average_loss,
        "perplexity": perplexity.item(),
        "inference_times": inference_times,
        "peak_memories": peak_memories,
        "average_memories": average_memories,
        "total_flops": total_flops,
        "flops_per_token": total_flops / total_tokens,
    }

In [13]:
!pip install matplotlib


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


In [14]:
import torch
from collections import defaultdict
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

def evaluate_expert_usage(model, test_dataloader, device):
    model.eval()
    total_tokens = 0

    metrics = {
        'expert_usage_per_token': [],
        'input_difficulties': [],
        'expert_distribution': defaultdict(int)
    }

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating Expert Usage"):
            input_ids = torch.stack(batch["input_ids"]).to(device)
            batch_tokens = input_ids.numel()
            total_tokens += batch_tokens

            outputs = model(input_ids=input_ids)

            total_expert_use = 0

            # Count expert activations
            for block in model.transformer.h:
                if isinstance(block, GPT2MoEBlock):
                    moe_layer = block.mlp
                    expert_weights = moe_layer.usage_counter  # [num_experts]
                    total_expert_use += expert_weights.sum().item()

                    for expert_id in range(len(expert_weights)):
                        metrics['expert_distribution'][expert_id] += expert_weights[expert_id].item()

                    moe_layer.usage_counter.zero_()  # Reset after each batch

            # Record average expert usage per token
            metrics['expert_usage_per_token'].append(total_expert_use / batch_tokens)

            # Input difficulty: entropy of softmax
            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
            metrics['input_difficulties'].append(entropy.mean().item())

    # Final Metrics
    metrics['avg_experts_per_token'] = np.mean(metrics['expert_usage_per_token'])
    metrics['std_experts_per_token'] = np.std(metrics['expert_usage_per_token'])

    if metrics['input_difficulties'] and metrics['expert_usage_per_token']:
        metrics['difficulty_expert_correlation'] = np.corrcoef(
            metrics['input_difficulties'],
            metrics['expert_usage_per_token']
        )[0, 1]

    total_usage = sum(metrics['expert_distribution'].values())
    if total_usage > 0:
        for expert_id in metrics['expert_distribution']:
            metrics['expert_distribution'][expert_id] /= total_usage

    # Print metrics
    print("\n📊 Expert Usage Metrics:")
    print(f"  - Average experts per token: {metrics['avg_experts_per_token']:.4f} ± {metrics['std_experts_per_token']:.4f}")
    if 'difficulty_expert_correlation' in metrics:
        print(f"  - Correlation (difficulty vs expert usage): {metrics['difficulty_expert_correlation']:.4f}")
    print("\nExpert Distribution:")
    for expert_id, usage in metrics['expert_distribution'].items():
        print(f"  - Expert {expert_id}: {usage:.2%}")

    # Plot expert distribution
    plt.figure(figsize=(10, 6))
    plt.bar(metrics['expert_distribution'].keys(), metrics['expert_distribution'].values())
    plt.title("Expert Usage Distribution")
    plt.xlabel("Expert ID")
    plt.ylabel("Usage Percentage")
    plt.show()

    # Plot difficulty vs usage
    plt.figure(figsize=(10, 6))
    plt.scatter(metrics['input_difficulties'], metrics['expert_usage_per_token'], alpha=0.5)
    plt.title("Input Difficulty vs Expert Usage")
    plt.xlabel("Input Difficulty (Entropy)")
    plt.ylabel("Experts per Token")
    plt.show()

    return metrics


In [30]:
def evaluate_expert_usage(model, test_dataloader, device):
    from collections import defaultdict
    model.eval()
    total_tokens = 0

    metrics = {
        'expert_usage_per_token': [],
        'input_difficulties': [],
        'expert_distribution': defaultdict(int),
        'expert_usage_by_layer': defaultdict(list)  # <=== new
    }

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc="Evaluating Expert Usage"):
            input_ids = torch.stack(batch["input_ids"]).to(device)
            batch_tokens = input_ids.numel()
            total_tokens += batch_tokens

            outputs = model(input_ids=input_ids)

            total_expert_use = 0

            for block in model.transformer.h:
                if isinstance(block, GPT2MoEBlock):
                    moe_layer = block.mlp
                    layer_id = moe_layer.layer_id
                    expert_weights = moe_layer.usage_counter  # [E]

                    # Sum usage for this layer
                    layer_expert_use = expert_weights.sum().item()
                    total_expert_use += layer_expert_use

                    # Log average usage per token in this layer
                    metrics['expert_usage_by_layer'][layer_id].append(layer_expert_use / batch_tokens)

                    for expert_id in range(len(expert_weights)):
                        metrics['expert_distribution'][(layer_id, expert_id)] += expert_weights[expert_id].item()

                    moe_layer.usage_counter.zero_()  # reset

            metrics['expert_usage_per_token'].append(total_expert_use / batch_tokens)

            logits = outputs.logits
            probs = F.softmax(logits, dim=-1)
            entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=-1)
            metrics['input_difficulties'].append(entropy.mean().item())

    metrics['avg_experts_per_token'] = np.mean(metrics['expert_usage_per_token'])
    metrics['std_experts_per_token'] = np.std(metrics['expert_usage_per_token'])

    if metrics['input_difficulties']:
        metrics['difficulty_expert_correlation'] = np.corrcoef(
            metrics['input_difficulties'],
            metrics['expert_usage_per_token']
        )[0, 1]

    # Normalize expert distribution
    total_usage = sum(metrics['expert_distribution'].values())
    if total_usage > 0:
        for k in metrics['expert_distribution']:
            metrics['expert_distribution'][k] /= total_usage

    # Print summary
    print(f"\n📊 Expert Usage Summary:")
    print(f"  - Average Experts/Token: {metrics['avg_experts_per_token']:.4f} ± {metrics['std_experts_per_token']:.4f}")
    if 'difficulty_expert_correlation' in metrics:
        print(f"  - Correlation (difficulty vs expert usage): {metrics['difficulty_expert_correlation']:.4f}")

    print("\n🔍 Per-Layer Expert Usage:")
    for layer_id in sorted(metrics['expert_usage_by_layer'].keys()):
        avg = np.mean(metrics['expert_usage_by_layer'][layer_id])
        print(f"  - Layer {layer_id:2d}: {avg:.4f} experts/token")

    return metrics


In [33]:
baseline_results = evaluate_model(baseline_model, test_dataloader, device)

Evaluating:   0%|          | 0/238 [00:00<?, ?it/s]/tmp/ipykernel_5058/450032448.py:36: FutureWarning: `use_cuda` is deprecated, use `activities` argument instead
  with profile(
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.
Evaluating: 100%|██████████| 238/238 [01:14<00:00,  3.20it/s]


🧪 Evaluation Results:
  - Average Loss: 9.9665
  - Perplexity: 21299.77
  - Total Inference Time: 61.69s
  - Average Time per Batch: 0.2592s
  - Average Time per Token: 0.0633 ms
  - Average Peak Memory per Batch: 4040.11 MB
  - Average Memory Used per Batch: 4040.05 MB
  - Total FLOPs: 1504981.58 GFLOPs
  - Average FLOPs per Token: 1545.43 MFLOPs


In [24]:
top_k_results=evaluate_model(top_k_model, test_dataloader, device)

Evaluating:   0%|          | 0/238 [00:00<?, ?it/s]/tmp/ipykernel_6179/450032448.py:36: FutureWarning: `use_cuda` is deprecated, use `activities` argument instead
  with profile(
Evaluating:   2%|▏         | 4/238 [07:22<7:11:09, 110.55s/it]


KeyboardInterrupt: 

In [31]:
top_k_metrics = evaluate_expert_usage(top_k_model, test_dataloader, device)

Evaluating Expert Usage: 100%|██████████| 238/238 [00:54<00:00,  4.41it/s]


📊 Expert Usage Summary:
  - Average Experts/Token: 18.0000 ± 0.0000
  - Correlation (difficulty vs expert usage): nan

🔍 Per-Layer Expert Usage:
  - Layer  0: 2.0000 experts/token
  - Layer  4: 2.0000 experts/token
  - Layer  8: 2.0000 experts/token
  - Layer 12: 2.0000 experts/token
  - Layer 16: 2.0000 experts/token
  - Layer 20: 2.0000 experts/token
  - Layer 24: 2.0000 experts/token
  - Layer 28: 2.0000 experts/token
  - Layer 32: 2.0000 experts/token



/opt/conda/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/conda/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [36]:
top_p_results=evaluate_model(top_p_model, test_dataloader, device)

Evaluating:   0%|          | 0/238 [00:00<?, ?it/s]/tmp/ipykernel_5058/450032448.py:36: FutureWarning: `use_cuda` is deprecated, use `activities` argument instead
  with profile(
Evaluating: 100%|██████████| 238/238 [02:44<00:00,  1.45it/s]


🧪 Evaluation Results:
  - Average Loss: 9.5840
  - Perplexity: 14529.81
  - Total Inference Time: 148.82s
  - Average Time per Batch: 0.6253s
  - Average Time per Token: 0.1528 ms
  - Average Peak Memory per Batch: 4054.24 MB
  - Average Memory Used per Batch: 4054.24 MB
  - Total FLOPs: 1917737.38 GFLOPs
  - Average FLOPs per Token: 1969.29 MFLOPs


In [39]:
top_p_metrics = evaluate_expert_usage(top_p_model, test_dataloader, device)

Evaluating Expert Usage:   0%|          | 0/238 [00:00<?, ?it/s]


AttributeError: 'MixtureOfExperts' object has no attribute 'layer_id'

In [ ]:
top_p_metrics = evaluate_expert_usage(top_p_model, test_dataloader, device)

Note that the y axis is experts per token per forward pass so across all the MoE layers

In [38]:
top_any_results=evaluate_model(top_any_model, test_dataloader, device)

Evaluating:   0%|          | 0/238 [00:00<?, ?it/s]/tmp/ipykernel_5058/450032448.py:36: FutureWarning: `use_cuda` is deprecated, use `activities` argument instead
  with profile(
Evaluating: 100%|██████████| 238/238 [03:07<00:00,  1.27it/s]


🧪 Evaluation Results:
  - Average Loss: 9.7699
  - Perplexity: 17499.61
  - Total Inference Time: 168.33s
  - Average Time per Batch: 0.7073s
  - Average Time per Token: 0.1729 ms
  - Average Peak Memory per Batch: 4057.76 MB
  - Average Memory Used per Batch: 4057.75 MB
  - Total FLOPs: 1708340.48 GFLOPs
  - Average FLOPs per Token: 1754.26 MFLOPs


In [39]:
top_any_metrics = evaluate_expert_usage(top_any_model, test_dataloader, device)

Evaluating Expert Usage:  27%|██▋       | 65/238 [00:14<00:39,  4.34it/s]


KeyboardInterrupt: 

In [ ]:
entropy_results=evaluate_model(entropy_model, test_dataloader, device)

In [ ]:
entropy_metrics = evaluate_expert_usage(entropy_model, test_dataloader, device)